<a href="https://colab.research.google.com/github/gitmystuff/DTSC3010/blob/main/Data_Science_Historical_Catastrophe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Science Historical Catastrophe

## Getting Started

* Colab - get notebook from gitmystuff DTSC3010 repository
* Save a Copy in Drive
* Remove Copy of
* Edit name
* Clean up Colab Notebooks folder
* Submit shared link

## Instructions

The goal of this assignment is to take messy, intentionally flawed data, clean it up, and analyze it using logistic regression — all framed around a historical catastrophe of your choosing.

**This is a "work on your own" assignment.** Follow each part in order. Do not skip ahead.

The workflow across all seven parts is:

| Part | Title | Your Role |
| :--- | :--- | :--- |
| **Part 1** | The Data | Run all cells and observe what is being built |
| **Part 2** | The Story | Choose a historical event and map the data variables to it |
| **Part 3** | Exploratory Data Analysis | Explore, visualize, and describe the data using your story names |
| **Part 4** | Data Prep | Clean the dataset — handle constants, duplicates, nulls, scaling, and outliers |
| **Part 5** | Feature Engineering | Create derived variables and encode categoricals |
| **Part 6** | Feature Selection | Use multiple selection methods to identify the best predictors |
| **Part 7** | Modeling and Evaluation | Build a logistic regression, evaluate it, and explain the results in writing |

> **On code cells with placeholder comments** (e.g., `# missing data`): these are your workspace. Write your solution directly in the cell beneath each comment. You have done this before — refer to your notes and previous notebooks for patterns.

> **On commented-out code cells**: uncomment and run them. They are provided as starting points or examples. Adapt column names to match your story mapping where needed.


# Part 1 - The Data

## Seed the Project

In [ ]:
import time
import numpy as np
import random

def generate_user_seed():
    # Get current time in nanoseconds (more granular)
    nanoseconds = time.time_ns()

    # Add a small random component to further reduce collision chances
    random_component = random.randint(0, 1000)  # Adjust range as needed

    # Combine them (XOR is a good way to mix values)
    seed = nanoseconds ^ random_component

    # Ensure the seed is within the valid range for numpy's seed
    seed = seed % (2**32)  # Modulo to keep it within 32-bit range

    return seed

user_seed = generate_user_seed()
print(user_seed)
random_state = np.random.seed(user_seed)

## Faker

In [ ]:
pip install Faker -q

In [ ]:
# rename this list so it is suitable to your story
habitable_planets = [
    "Alpha Centauri III",
    "Eden",
    "Terra Nova",
    "Tiberius",
    "Vega Colony",
    "Cait",
    "Andoria",
    "Vulcanis",
    "Risa",
    "Betazed",
    "Ba'ku",
    "Aldea",
    "Nimbus III",
    "Deneva",
    "Capella IV",
    "Organia",
    "Trillius Prime",
    "Kaelon II",
    "Mintaka III",
    "Rubicun III",
    "Pacifica",
    "Tau Ceti III",
    "Melina",
    "Argelius II",
    "Iconia",
    "Alderaan",
    "Naboo",
    "Bespin (Cloud City)",
    "Yavin IV",
    "Endor (Forest Moon)",
    "Kashyyyk",
    "Mon Cala",
    "Corellia",
    "Chandrila",
    "Ryloth",
    "Cato Neimoidia",
    "Felucia",
    "Saleucami",
    "Stewjon",
    "Iego",
    "Glee Anselm",
    "Mirial",
    "Serenno",
    "Malastare",
    "Dantooine",
    "Haruun Kal",
    "Manaan",
    "Zolan",
    "Ord Mantell",
    "Pantora"
]

In [ ]:
# create demographic data
import numpy as np
import pandas as pd
from faker import Faker
fake = Faker()

n = 1000

output = []
for x in range(n):
    biology = np.random.choice(['Cytophore', 'Kymete'], p=[0.5, 0.5])
    output.append({
        'categorical_1': biology,
        'categorical_2': np.random.choice(['Xylosian', 'Veridian', 'CKaeltharr']),
        'name_1': fake.first_name_female() if biology == 'Cytophore' else fake.first_name_male(),
        'name_2': fake.last_name(),
        'code': fake.zipcode(),
        'date': fake.date_of_birth(),
        'location': np.random.choice(habitable_planets)
    })

demographics = pd.DataFrame(output)
print(demographics.shape)
demographics.head()

## Create Independent Variable Correlated with Class

In [ ]:
import numpy as np
import pandas as pd

def generate_feature(df, class_col, coeff, intercept):
    """
    Generates normally distributed feature data for a logistic regression model.

    Args:
        df: The pandas DataFrame containing the class column.
        class_col: The name of the class column (containing 0s and 1s).
        coeff: The coefficient for the feature in the logistic regression model.
        intercept: The intercept of the logistic regression model.

    Returns:
        A pandas Series containing the generated feature data.
    """

    # Generate probabilities based on the class
    probs = np.random.rand(len(df))  # Initial random probabilities
    probs = np.where(df[class_col] == 1, probs * 0.8 + 0.2, probs * 0.8)  # Adjust for class

    # Apply the inverse logit (logit) function
    logits = np.log(probs / (1 - probs))

    # Calculate the feature values
    feature_values = (logits - intercept) / coeff

    return pd.Series(feature_values)



## Make Classification

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression

def make_linear_y(row):
  model = LogisticRegression()
  model.fit(X, y)
  coefficients = model.coef_
  intercept = model.intercept_
  f_of_x = intercept + coefficients[0][0]*row['informative_1'] + coefficients[0][1]*row['informative_2']
  # print(f_of_x[0])
  return f_of_x[0]

# Adjust the make_classification parameters:
# Set n_informative and n_redundant to values that sum to less than n_features
X, y = make_classification(n_samples=n, n_features=2, n_informative=2, n_redundant=0, n_repeated=0, random_state=42)
df = pd.DataFrame(X, columns=['informative_1', 'informative_2'])
df = pd.concat([demographics, df], axis=1).reset_index(drop=True)

df['target'] = df.apply(make_linear_y, axis=1) # an independent variable
df['class'] = y # the dependent variable
df['corr_feature_class'] = generate_feature(df, 'class', 0.5, -1)
df.head()

## Automation Functions

1. gen_null(series, perc)
2. gen_quasi_constants(primary_label, variation_percentage=.2, size=len(df))
3. gen_normal_data(mu=0, std=1, size=len(df))
4. gen_uniform_data(size=len(df))
5. gen_multivariate_normal_data(mean=[0, 0], cov=[[1, 0], [0, 1]], size=len(df))
6. gen_correlated_normal_series(original_series, target_correlation, size=len(df))
7. gen_correlated_uniform_series(original_series, correlation_coefficient=0, size=len(df))
8. gen_outliers(mean=0, std_dev=1, size=len(df), outlier_percentage=0.1, outlier_magnitude=3)
9. gen_standard_scaling(mean=50, std_dev=10, size=len(df), scale_factor=1000)
10. gen_minmax_scaling(mean=50, std_dev=10, size=len(df), range_factor=10)
11. random_choice_data(choices, size)

In [ ]:
# functions
import pandas as pd
import numpy as np
from scipy.stats import norm
from scipy.optimize import minimize


def gen_null(series, perc):
  """
  Introduces null values (np.nan) into a list based on a specified percentage.

  Args:
      var: The variable to modify.
      perc: The percentage of values to replace with nulls (0-100).

  Returns:
      The modified variable with null.
  """
  var = series.copy()
  num_nulls = int(len(var) * (perc / 100))
  indices_to_replace = np.random.choice(len(var), num_nulls, replace=False)

  for idx in indices_to_replace:
      var[idx] = np.nan

  return var

def gen_quasi_constants(primary_label, variation_percentage=.2, size=len(df)):
  """
  Generates quasi-constant labels for a Series, with a small percentage of variation.

  Args:
      primary_label: The main label to use for most values.
      variation_percentage: The percentage of labels to vary (0-100).

  Returns:
      A new Series containing the quasi-constant labels.
  """

  series = pd.Series(np.full(size, primary_label))
  num_variations = int(size * (variation_percentage / 100))
  variation_indices = np.random.choice(series.index, num_variations, replace=False)
  primary_label = primary_label + '_0'
  variation1 = primary_label + '_1'
  variation2 = primary_label + '_2'

  labels = pd.Series([primary_label] * len(series), index=series.index)
  labels.loc[variation_indices] = np.random.choice([variation1, variation2], size=num_variations)  # Adjust variations as needed

  return labels

def gen_normal_data(mu=0, std=1, size=len(df)):
  """
  Generates a normal dataset given the mean and standard deviation

  Args:
        mu: The mean of the normal distribution.
        std: The standard deviation of the normal distribution.
        size: The number of data points to generate.

  Returns:
        A normally distributed series.
  """
  return np.random.normal(mu, std, size)

def gen_uniform_data(size=len(df)):
  """
  Generates a uniform dataset

  Args:
        size: The number of data points to generate.

  Returns:
        A uniform distributed series.
  """
  return np.random.uniform(size=size)

def gen_multivariate_normal_data(mean=[0, 0], cov=[[1, 0], [0, 1]], size=len(df)):
  """
  Generates two datasets with a multivariate normal distribution given the mean and covariance matrix

  Args:
        mean: The mean of each of the datasets.
        cov: The covariance matrix of the datasets.
        size: The number of data points to generate.

  Returns:
        Two correlated series.
  """
  ds1, ds2 = np.random.multivariate_normal(mean, cov, size, tol=1e-6).T # ds = dataset
  return ds1, ds2

def gen_correlated_normal_series(original_series, target_correlation, size=len(df)):
  """
  Generates a correlated series based on a given series.

  This function takes an original series as input and generates a new series
  that is correlated with the original series. The correlation between the
  original and generated series is approximately equal to the specified
  target correlation.

  The generated series is created by linearly transforming the original series
  and adding Gaussian noise with an adjusted standard deviation to achieve the
  desired correlation.

  Args:
      original_series (numpy.ndarray): The original series.
      target_correlation (float): The desired Pearson correlation coefficient
          between the original and generated series.

  Returns:
      numpy.ndarray: The generated correlated series.
  """
  return np.mean(original_series) + target_correlation * (original_series - np.mean(original_series)) \
  +  np.random.normal(0, np.sqrt(1 - target_correlation**2) * np.std(original_series), len(original_series))
  """
  Explanation

  This one-liner leverages the properties of linear transformations and normal distributions to generate a correlated series.

  It first centers the original_series by subtracting its mean.
  It then scales this centered series by the target_correlation.
  Finally, it adds Gaussian noise with a standard deviation adjusted to ensure the overall correlation matches the target_correlation.
  """

def gen_correlated_uniform_series(original_series, correlation_coefficient=0, size=len(df)):
  """
  Work in progress

  Generates a new series correlated with the given series based on the specified correlation coefficient,
  using rank correlation to ensure the generated series follows a uniform distribution.

  Args:
      original_series (numpy.ndarray or list): The original series.
      correlation_coefficient (float): The desired correlation coefficient between the original and generated series.
      size: The number of data points to generate.

  Returns:
      The generated correlated series with a uniform distribution.
  """
  z_scores = (original_series - np.mean(original_series)) / np.std(original_series)
  correlation_coefficient=.7
  return norm.cdf(correlation_coefficient * norm.ppf(np.random.uniform(size=size)) + np.sqrt(1 - correlation_coefficient**2) * z_scores)

def pearson_r_func(x, y, y_mean, y_std, desired_r):
    x_mean = np.mean(x)
    x_std = np.std(x)
    numerator = np.sum((x - x_mean) * (y - y_mean))
    denominator = x_std * y_std * len(x)
    calculated_r = numerator / denominator
    return (calculated_r - desired_r)**2  # Minimize the squared difference

def minimize_r(original_series, target_correlation, size=len(df)):
    y = original_series
    y_mean = np.mean(y)
    y_std = np.std(y)
    desired_r = target_correlation

    # Initial guess for x values
    x0 = np.random.uniform(size=len(original_series))

    # Solve for x
    result = minimize(pearson_r_func, x0, args=(y, y_mean, y_std, desired_r))

    if result.success:
        x_solution = result.x
        # print("Solution for x:", x_solution)
        return x_solution
    else:
        print("Optimization failed.")

def gen_outliers(mean=0, std_dev=1, size=len(df), outlier_percentage=0.1, outlier_magnitude=3):
    """
    Generates a normal distribution with outliers.

    Args:
        mean (float): The mean of the normal distribution.
        std_dev (float): The standard deviation of the normal distribution.
        size (int): The number of samples to generate.
        outlier_percentage (float): The percentage of outliers to introduce (between 0 and 1).
        outlier_magnitude (float): The magnitude by which outliers deviate from the mean.

    Returns:
        numpy.ndarray: The generated data with outliers.
    """
    data = np.random.normal(mean, std_dev, size)
    num_outliers = int(size * outlier_percentage)
    outlier_indices = np.random.choice(size, num_outliers, replace=False)
    for index in outlier_indices:
        if np.random.rand() < 0.5:
            data[index] += outlier_magnitude
        else:
            data[index] -= outlier_magnitude

    return data

def gen_standard_scaling(mean=50, std_dev=10, size=len(df), scale_factor=1000):
  """
  Generates data with a specified mean and standard deviation, then scales it by a factor to create a distribution needing scaling.

  Args:
      mean (float): The mean of the original distribution.
      std_dev (float): The standard deviation of the original distribution.
      size (int): The number of samples to generate.
      scale_factor (float): The factor by which to scale the original distribution.

  Returns:
      numpy.ndarray: The generated data needing scaling.
  """
  original_data = np.random.normal(mean, std_dev, size)
  return original_data * scale_factor

def gen_minmax_scaling(mean=50, std_dev=10, size=len(df), range_factor=10):
  """
  Generates data with a specified mean and standard deviation, then scales and shifts it to create a distribution needing MinMax scaling.

  Args:
      mean (float): The mean of the original distribution.
      std_dev (float): The standard deviation of the original distribution.
      size (int): The number of samples to generate.
      range_factor (float): The factor to expand the range of the original distribution.

  Returns:
      numpy.ndarray: The generated data needing scaling.
  """

  # Generate the original data
  original_data = np.random.normal(mean, std_dev, size)

  # Expand the range of the data
  min_val = np.min(original_data)
  max_val = np.max(original_data)
  return (original_data - min_val) * range_factor + min_val

def random_choice_data(choices, size):
  """
  Generates a new series correlated with the given series based on the specified correlation coefficient,
  using rank correlation to ensure the generated series follows a uniform distribution.

  Args:
      original_series (numpy.ndarray or list): The original series.
      correlation_coefficient (float): The desired correlation coefficient between the original and generated series.

  Returns:
      numpy.ndarray: The generated correlated series with a uniform distribution.
  """
  return np.random.choice(choices, size=size)


In [ ]:
# categorical variables with little correlation to target
df['random choice 2'] = random_choice_data(['Rand Choice 1', 'Rand Choice 2'], size=len(df))
df['random choice 4'] = random_choice_data(['North', 'South', 'East', 'West'], size=len(df))
df['random choice 7'] = random_choice_data(['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'], size=len(df))

# categorical random choices with random # of labels
num_labels = np.random.randint(3, 5)
df[f'random label num {num_labels}'] = random_choice_data([f'label num lo {i}' for i in range(1, num_labels + 1)], size=len(df))

num_labels = np.random.randint(10, 15)
df[f'random label num {num_labels}'] = random_choice_data([f'label num hi {i}' for i in range(1, num_labels + 1)], size=len(df))

In [ ]:
# categorical variables correlated with target
df['pd qcut1'] = pd.qcut(df['target'], 2, labels=['Low', 'High']) # bi label
df['pd qcut2'] = pd.qcut(df['target'], 4, labels=['Q1', 'Q2', 'Q3', 'Q4']) # 4 labels

quantiles = [0, 0.1, 0.2, 0.4, 0.6, 0.8, 1]
df['pd qcut3'] = pd.qcut(df['target'], quantiles, labels=['G1', 'G2', 'G3', 'G4', 'G5', 'G6']) # 6 labels

In [ ]:
# generate four numerical normally distributed continuous features that have a correlation greater than absolute value of .5 with each other
# gen_multivariate_normal_data(mean=[0, 0], cov=[[1, 0], [0, 1]], size=len(df))
df['multicollinearity 1'], df['multicollinearity 2'] = gen_multivariate_normal_data(mean=[0, 0], cov=[[1, .7], [.7, 1]], size=len(df))
df['multicollinearity 3'], df['multicollinearity 4'] = gen_multivariate_normal_data(mean=[0, 0], cov=[[1, .9], [.9, 1]], size=len(df))

In [ ]:
# generate two normally distributed features that are correlated with the target
# gen_correlated_normal_series(original_series, target_correlation, size=len(df))
df['correlated w target 1'] = gen_correlated_normal_series(df['target'], target_correlation=.5)
df['correlated w target 2'] = gen_correlated_normal_series(df['target'], target_correlation=.7)
df.info()

In [ ]:
# generate two uniformly distributed features that are correlated with the target
# gen_correlated_uniform_series(original_series, correlation_coefficient=0, size=len(df))
df['uniform corr 1'] = gen_correlated_uniform_series(df['target'])
df['uniform corr 2'] = gen_correlated_uniform_series(df['target'])

In [ ]:
# create two features that are duplicates of other features
df['duplicate_1'] = df['informative_1']
df['duplicate_2'] = df['informative_2']

In [ ]:
# create two numerical features with outliers
df['outliers 1'] = gen_outliers(mean=0, std_dev=1, size=len(df), outlier_percentage=0.1, outlier_magnitude=3)
df['outliers 2'] = gen_outliers(mean=3, std_dev=2, size=len(df), outlier_percentage=0.2, outlier_magnitude=2)

In [ ]:
# create a numerical feature that needs standard scaling
df['standard scaling'] = gen_standard_scaling()

In [ ]:
# create a numerical feature that needs min max scaling
df['min max scaling'] = gen_minmax_scaling()

In [ ]:
# generate null values
for col in df.drop(['class', 'informative_1', 'informative_2', 'target', 'duplicate_1', 'duplicate_2'], axis=1).columns:
    df[col] = gen_null(df[col], np.random.choice([0, 5, 10, 20, 30, 50], size=1).item())

In [ ]:
# create two features that have constant values
df['constant_1'] = 'constant_value'
df['constant_2'] = 'constant_value'

In [ ]:
# create two features with semi constant values
df['semi_constant_1'] = gen_quasi_constants('q_const', variation_percentage = 1)
df['semi_constant_2'] = gen_quasi_constants('q_const', variation_percentage = 1)

In [ ]:
print(df.info())  # check progress

In [ ]:
# add duplicates
dupes = df.loc[0:9]
df = pd.concat([df, dupes], axis=0)

# shuffle all columns
# df = df.sample(frac=1).reset_index(drop=True)
# df = df.sample(frac=1, axis=1)

# shuffle selected columns
demographic_columns = demographics.columns
remaining_columns = [col for col in df.columns if col not in demographic_columns]
# print(remaining_columns)
np.random.shuffle(remaining_columns)

# Reassemble the DataFrame with the shuffled columns
df = df[list(demographic_columns) + list(remaining_columns)]

# move target to the end of the list
class_var = 'class'
df = df[df.drop('class', axis=1).columns.tolist() + [class_var]]

print(df.shape)
print(df.info())
df.head()

In [ ]:
df.to_csv('data science fiction iv pt 1.csv', index=False)

# Part 2 - The Story

Using historical catastrophes for a data science assignment is a fantastic way to make feature engineering and selection feel "high stakes." With 40 variables, you have enough room to distinguish between environmental factors, human error, and systemic failures.

Here are some examples:

---

## 1. The Great Smog of London (1952)
This event is perfect for a **Public Health & Environmental** narrative. Students can model the likelihood of a "high-mortality event" across different London boroughs.

* **The Binary Target:** High Mortality vs. Baseline Mortality.
* **Variable Mapping Ideas:**
    * **Environmental:** Humidity, temperature inversions, wind speed, sulfur dioxide levels.
    * **Socio-economic:** Average age of residents, coal usage per household, proximity to industrial Thames-side factories.
    * **Infrastructure:** Hospital bed capacity, accessibility to public transport (trams vs. buses).
* **The "Story" Hook:** Students act as early epidemiologists trying to prove that the deaths were caused by pollution rather than a sudden flu epidemic.

---

## 2. The Bronze Age Collapse (c. 1177 BC)
For a more **Geopolitical & Archaeological** twist, students can analyze various settlements across the Mediterranean to predict which ones survived and which were destroyed.

* **The Binary Target:** Settlement Collapse vs. Settlement Survival.
* **Variable Mapping Ideas:**
    * **Trade:** Proximity to copper mines, distance from tin trade routes, number of foreign artifacts found.
    * **Geography:** Elevation, distance to coast (vulnerability to "Sea Peoples"), soil acidity/crop yield.
    * **Conflict:** Thickness of city walls, presence of arrowheads in the strata, frequency of fires.
* **The "Story" Hook:** As "Digital Archaeologists," students are identifying the primary "stressor" (Famine? Earthquake? Invasion?) that led to the end of the Mycenaean or Hittite empires.

---

## 3. The Great Fire of Rome (64 AD)
This fits a **Crisis Management & Urban Planning** narrative. Students can analyze "Insulae" (apartment blocks) to determine the probability of total destruction.

* **The Binary Target:** Total Incineration vs. Surviving Structure.
* **Variable Mapping Ideas:**
    * **Construction:** Wood vs. stone ratios, height of the building, width of the surrounding street (firebreaks).
    * **Location:** Proximity to the Circus Maximus (where the fire started), elevation (Palatine Hill vs. the Subura).
    * **Resources:** Number of "Vigiles" (firefighters) nearby, cistern capacity, occupancy density.
* **The "Story" Hook:** Students are investigating the disaster to determine if the fire was an accident of urban overcrowding or a deliberate act of arson (addressing the Nero legends).

---

### Hints:
* **Feature Engineering:** Create "Risk Scores" by combining environmental and infrastructure variables.
* **Feature Selection:** With 40 variables, some should be "noise" (e.g., the name of the local magistrate) while others are highly correlated (e.g., temperature and wind speed), forcing them to handle multicollinearity.
* **The "Messy" Factor:** Historical data is naturally incomplete. This is simulated by introducing "missing" values for variables like "Ancient Crop Yield" or "1950s Air Quality Sensor Data."

Example Prompt

AI, here is an example of a dataset i am suppose to generate a historical catastrophic story about. I am using a classification model so my output will be discreet.

Data columns (total 38 columns): # Column Non-Null Count Dtype --- ------ -------------- ----- 0 categorical_1 1010 non-null object 1 categorical_2 506 non-null object 2 name_1 809 non-null object 3 name_2 809 non-null object 4 code 909 non-null float64 5 date 1010 non-null object 6 location 810 non-null object 7 duplicate_2 1010 non-null float64 8 standard scaling 508 non-null float64 9 informative_1 1010 non-null float64 10 random choice 2 506 non-null object 11 semi_constant_2 1010 non-null object 12 multicollinearity 4 960 non-null float64 13 random choice 7 809 non-null object 14 uniform corr 1 504 non-null float64 15 constant_1 1010 non-null object 16 informative_2 1010 non-null float64 17 corr_feature_class 960 non-null float64 18 pd qcut3 506 non-null object 19 pd qcut1 509 non-null object 20 semi_constant_1 1010 non-null object 21 correlated w target 1 1010 non-null float64 22 uniform corr 2 1010 non-null float64 23 random label num 4 707 non-null object 24 pd qcut2 910 non-null object 25 multicollinearity 2 707 non-null float64 26 multicollinearity 1 910 non-null float64 27 duplicate_1 1010 non-null float64 28 multicollinearity 3 1010 non-null float64 29 correlated w target 2 1010 non-null float64 30 min max scaling 1010 non-null float64 31 constant_2 1010 non-null object 32 outliers 1 1010 non-null float64 33 outliers 2 809 non-null float64 34 random choice 4 809 non-null object 35 target_perfect 1010 non-null float64 36 random label num 10 960 non-null object 37 target 1010 non-null float64



## Your Story

**Step 1 — Choose a scenario.** Pick a historical catastrophe. Be creative. You may use one of the three examples above (Great Smog, Bronze Age Collapse, Great Fire of Rome) or propose your own. Your scenario must have a clear binary outcome that maps to `class = 0` (normal) and `class = 1` (catastrophic event).

**Step 2 — Generate your story using AI.** Copy the example prompt below and paste it into an AI assistant (e.g., Claude, ChatGPT). Use the actual `df.info()` output from your Part 1 run — not the sample output shown above, since column names are shuffled randomly each time.

**Step 3 — Paste the AI's response below.** Replace this instruction block with your story title and narrative. Your story must include:
- The historical setting and what is being measured
- A clear definition of `class = 0` (H₀, the normal state)
- A clear definition of `class = 1` (H₁, the catastrophic event)
- At least one paragraph of narrative context

---

Read the next cell for more context.

## Things to Think About: Historical Catastrophe Edition

Your story provides the context, but your data provides the evidence. Combining both creates a stronger analytical narrative. In historical data science, we often use records, measurements, and observations to determine whether a disaster is unfolding—or whether people are simply reacting to ordinary variation.

### 1. Identify the “Status Quo” (H₀)

In historical catastrophe analysis, the Null Hypothesis (H₀) represents the normal state of society or environment. It assumes that nothing extraordinary is happening.

Narrative Example:
“The increase in deaths this month is part of normal seasonal illness.”

Data Mapping:
H₀: Class = 0 (Normal Conditions)

Examples:

* Grain shortages are due to regular seasonal fluctuations.
* River levels are within ordinary annual variance.
* Earth tremors are natural background activity.
* Disease cases reflect expected patterns.

### 2. Identify the “Crisis” (H₁)

The Alternative Hypothesis (H₁) represents the disruptive event—the catastrophe.

Narrative Example:
“The increase in deaths indicates the beginning of a plague outbreak.”

Data Mapping:
H₁: Class = 1 (Catastrophic Event)

Examples:

* Famine is beginning.
* A plague is spreading.
* A flood is imminent.
* An army is advancing.
* A volcano is preparing to erupt.

### 3. Establish the “Decision Rule” (Threshold)

The decision rule determines when leaders act.

In data science, this is the probability threshold where evidence becomes strong enough to reject H₀.

Questions:

* When do city officials quarantine?
* When do rulers ration grain?
* When do villages evacuate?
* When do armies mobilize?

The Task:
Define what “success” means.

Is it more important to:

* Catch every possible plague outbreak? (High Recall)
* Avoid unnecessary panic and false alarms? (High Precision)

### 4. Risk Assessment (Error Analysis)

Map statistical errors to story consequences.

## The Historical Threat Matrix

| Topic                  | Safe Null Hypothesis (H₀)               | Catastrophic Alternative (H₁)                    |
| ---------------------- | --------------------------------------- | ------------------------------------------------ |
| The Failing Harvest    | Low crop yields are seasonal variation  | A famine has begun                               |
| The Rising River       | Water levels are normal spring swelling | A catastrophic flood is imminent                 |
| The Strange Illness    | Villagers have ordinary fever           | A plague outbreak has started                    |
| The Earth Tremors      | Minor geological movement               | A volcanic eruption is coming                    |
| The Silent Border      | Missing traders are delayed by weather  | An invading army is approaching                  |
| The Empty Granaries    | Distribution errors caused shortages    | Systemic crop failure threatens the kingdom      |
| The Dying Livestock    | Animal deaths are isolated disease      | A widespread agricultural collapse is underway   |
| The Darkened Sky       | Dust clouds are weather-related         | Volcanic ash signals a major eruption            |
| The Falling Population | Migration explains missing workers      | Mass death has begun                             |
| The Broken Trade Route | Roads are temporarily blocked           | Political collapse or war has disrupted commerce |

## The Statistical Stakes

In historical crises, leaders make decisions with incomplete data.

### High Precision (α → 0)

“Do not declare plague unless we are absolutely certain. Panic could destroy the economy.”

Goal:
Avoid false positives.

Risk:
Miss the actual disaster.

### High Recall (β → 0)

“Seal the city gates immediately. Even a small chance of plague is too dangerous.”

Goal:
Catch every threat.

Risk:
Overreact to harmless variation.

Decision formula:

P(\text{Catastrophe}\mid\text{Data})>\tau

Where τ is the action threshold.

If the probability exceeds τ, leaders abandon the status quo and act.

## Type I Error (α): The False Alarm

Statistical Definition:
Rejecting H₀ when H₀ is true.

Historical Translation:
Treating a normal situation as a catastrophe.

### Example: The Failing Harvest

Reality (H₀):
Crop yields are within normal variation.

Action (H₁):
The king declares famine and seizes grain.

Consequence:
Markets collapse, food prices surge, and unrest begins.

Hero’s Regret:
“We created the crisis ourselves.”

### Example: The Strange Illness

Reality (H₀):
A seasonal flu.

Action (H₁):
The city enforces quarantine.

Consequence:
Trade stops, wages vanish, and riots erupt.

Hero’s Regret:
“We panicked too soon.”

## Type II Error (β): The Silent Killer

Statistical Definition:
Failing to reject H₀ when H₁ is true.

Historical Translation:
Treating catastrophe as normal.

### Example: The Strange Illness

Reality (H₁):
The plague has begun.

Action (H₀):
Officials assume it is routine illness.

Consequence:
The disease spreads unchecked.

Hero’s Regret:
“The warning signs were there.”

### Example: The Rising River

Reality (H₁):
A flood is imminent.

Action (H₀):
Leaders ignore the river data.

Consequence:
The city is destroyed overnight.

Hero’s Regret:
“We waited too long.”

## Summary Table: The Narrative Cost of Error

| Error Type  | Statistical Term | Story Impact                                  | The Leader’s Regret  |
| ----------- | ---------------- | --------------------------------------------- | -------------------- |
| Type I (α)  | False Positive   | Overreaction wastes resources or causes panic | “We acted too soon.” |
| Type II (β) | False Negative   | Failure to act allows catastrophe             | “We acted too late.” |

## Final Writing Prompt

Choose one historical catastrophe scenario.

Build your story around:

1. What is the normal explanation (H₀)?
2. What is the catastrophic explanation (H₁)?
3. What data is being observed?
4. What threshold triggers action?
5. What happens if leaders make a Type I error?
6. What happens if leaders make a Type II error?

Your goal is to tell a compelling historical story while demonstrating an understanding of classification, hypothesis testing, and error analysis.


## Paste Your Story (AI Response) Here

## Variable Mapping

The dataset contains system-generated variable names. Your job is to rename every variable so it fits your historical catastrophe story. You are not changing the data — you are giving it narrative meaning.

**Step 1 — Complete the mapping table below.** Every column in your dataset must appear in the table. Use your `df.info()` output from Part 1 as your column list.

**Step 2 — Be consistent.** The names you assign here must be used throughout Parts 3–7. When the instructions reference a generic name like `informative_1`, use your renamed version instead.

**Rules:**
- Every renamed variable must fit your story logically.
- Do not rename things randomly — new names must have clear meaning.
- Pay special attention to: `date`, `location`, `target`, `informative_1`, `informative_2`, `correlated w target 1`, `correlated w target 2`, and `class`.

**Example — Plague Outbreak Story:**

| Original Variable Name | Story Variable Name | Why It Fits |
| :--- | :--- | :--- |
| `date` | `date_of_symptoms` | Date the patient was first recorded |
| `location` | `infected_village` | The settlement where the case was reported |
| `target` | `fever_intensity` | Continuous measure of illness severity |
| `class` | `plague_event` | 0 = routine illness, 1 = confirmed plague case |
| `informative_1` | `crowding_index` | Population density — a key transmission factor |
| `outliers_1` | `extreme_mortality_rate` | Unusually high death counts in a district |

---

**Your Mapping Table:**

| Original Variable Name | Story Variable Name | Why It Fits |
| :--- | :--- | :--- |
| | | |

*Add as many rows as you have columns.*


# Part 3 - Exploratory Data Analysis (EDA)

Exploratory data analysis (EDA) is a data analysis method that helps data scientists understand their data and identify patterns. It's often used as the first step in data analysis.

## Load Data

In [ ]:
import pandas as pd

df = pd.read_csv('data science fiction iv pt 1.csv')
print(df.shape)
print(df.info())
df.head()

## Var Types

Identifying variable types is the "measure twice, cut once" phase of data science. Before you can build a model or even create a simple plot, you have to know what kind of data you’re holding.

Here is why this step is foundational:

### 1. Choosing the Right Statistical Strategy
Statistical tests are picky. You cannot perform the same math on a zip code (categorical) as you would on a salary (numerical).
* **Numerical Data:** You can calculate the mean, variance, and standard deviation.
* **Categorical Data:** These operations are meaningless. Instead, you look at mode, frequency, and proportions.

### 2. Directing Exploratory Data Analysis (EDA)
Your variable types dictate your visualizations. If you pick the wrong chart for your data type, you’ll likely end up with a confusing mess rather than an insight.
* **Continuous variables** (like temperature) call for histograms or box plots to see distribution.
* **Discrete/Categorical variables** (like car brand) call for bar charts or pie charts to see counts.



### 3. Requirements for Machine Learning
Most machine learning algorithms are essentially giant calculators—they only understand numbers.
* **Encoding:** If you have categorical data (e.g., "Red," "Blue," "Green"), you must identify them so you can apply techniques like **One-Hot Encoding** or **Label Encoding** to turn them into a format the model can process.
* **Feature Scaling:** Certain models (like K-Nearest Neighbors or SVMs) are sensitive to the scale of numerical data. You need to identify these variables to apply normalization or standardization.

### 4. Data Cleaning and Error Detection
Identifying types helps you spot "dirty" data quickly. If a column labeled "Age" suddenly contains the string "Unknown" or a negative number, knowing that the variable *should* be a positive integer allows you to set up automated validation rules to catch these outliers.

In [ ]:
df_numerical = df.select_dtypes(include='number').columns
df_object = df.select_dtypes(include=['object']).columns
df_discreet = df.select_dtypes(include=['category']).columns
df_categorical_features = df.select_dtypes(include=['category', 'object']).columns
print(df_numerical)
print(df_object)
print(df_discreet)
print(df_categorical_features)

### ✏️ Your Turn — Variable Type Visualizations

The two code cells below are commented out. Uncomment them and run them to visualize your variable distributions. Replace the column names with your own story variable names. After running, add a brief comment noting what you observe (e.g., is the variable balanced? skewed? dominated by one label?).


In [ ]:
# import matplotlib.pyplot as plt
# import seaborn as sns

# plt.figure(figsize=(10, 6))
# sns.histplot(df['target'], kde=True, color='blue')
# plt.title('Distribution of Target Variable')
# plt.xlabel('Target Value')
# plt.ylabel('Frequency')
# plt.show()

In [ ]:
# plt.figure(figsize=(10, 6))
# sns.countplot(x='categorical_1', data=df, order=df['categorical_1'].value_counts().index)
# plt.title('Frequency of Categorical_1 Levels')
# plt.xticks(rotation=45)
# plt.show()

## Correlation

Correlation analysis is a statistical method used to evaluate the strength and direction of the relationship between two quantitative variables. In EDA, it serves several critical purposes:


### 1. Identifying Feature Redundancy (Multicollinearity)
If two independent variables are highly correlated, they provide redundant information. This can lead to **Multicollinearity**, which inflates the variance of coefficient estimates in models like Linear Regression, making the model unstable.

* **Pearson Correlation Coefficient ($\rho$):** Measures linear relationship strength.
    $$\rho_{X,Y} = \frac{\text{cov}(X,Y)}{\sigma_X \sigma_Y}$$
    Where values range from $-1$ (perfect negative) to $+1$ (perfect positive).

### 2. Feature Selection and Target Relationship
EDA uses correlation to identify which features ($X$) have the most significant impact on the target variable ($y$).
* High $|\rho|$ suggests a strong predictor.
* Low $|\rho|$ might suggest a variable can be dropped, or that the relationship is non-linear.



### 3. Detecting Non-Linear Patterns
A common pitfall is assuming a low correlation means "no relationship." EDA combines correlation coefficients with **Scatter Plots** to spot non-linear patterns (e.g., quadratic or exponential) that $\rho$ cannot capture.


### 4. Data Understanding and Directionality
Correlation tells us the nature of the "link" between variables:
* **Positive Correlation ($\rho > 0$):** Variables move in the same direction.
* **Negative Correlation ($\rho < 0$):** Variables move in opposite directions.


In [ ]:
# Uncomment and run to view the raw correlation matrix for all numerical columns.
# This gives you a first look at which features move together.
df[df_numerical].corr().round(2)

### ✏️ Your Turn — Correlation Heatmap and Feature Pair Analysis

Uncomment and run the two cells below to generate a correlation heatmap and identify highly correlated feature pairs. Make note of any pairs with |ρ| > 0.7 — you will need to address those in Part 4.


In [ ]:
# # show correlation between the features
# import numpy as np
# import matplotlib.pyplot as plt
# import seaborn as sns

# # correlation matrix
# sns.set(style="white")

# # compute the correlation matrix
# corr = df[df_numerical].corr().round(1)

# # generate a mask for the upper triangle
# mask = np.zeros_like(corr, dtype=bool)
# mask[np.triu_indices_from(mask)] = True

# # set up the matplotlib figure
# # f, ax = plt.subplots()
# f = plt.figure(figsize=(12, 12))

# # generate a custom diverging colormap
# cmap = sns.diverging_palette(220, 10, as_cmap=True)

# # draw the heatmap with the mask and correct aspect ratio
# sns.heatmap(corr, mask=mask, cmap=cmap, vmax=.3, center=0,
#             square=True, linewidths=.5, cbar_kws={"shrink": .5}, annot=True);

# plt.tight_layout()

In [ ]:
# # calculate the correlation matrix
# corr_matrix = df[df_numerical].corr()

# # Create a mask for the upper triangle (to avoid duplicates)
# mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

# # Convert the correlation matrix to a long format
# corr_df = corr_matrix.stack().reset_index()
# corr_df.columns = ['feature1', 'feature2', 'correlation']

# # Filter for correlations above a certain threshold (e.g., 0.7)
# high_corr_df = corr_df[(abs(corr_df['correlation']) > 0.7) & (corr_df['feature1'] != corr_df['feature2'])]

# # Sort by absolute correlation in descending order
# high_corr_df = high_corr_df.sort_values(by='correlation', ascending=False, key=abs)

# # Print the top correlated features
# # print(high_corr_df['feature1'].to_list()[4:10])
# print(high_corr_df)

# # Create a variable to pickle
# data = {'correlation scores': high_corr_df}

In [ ]:
# # check for vif
# import pandas as pd
# import numpy as np
# from sklearn.preprocessing import StandardScaler
# from statsmodels.stats.outliers_influence import variance_inflation_factor

# # handle null values (using mean imputation for simplicity)
# x_copy = df.drop('class', axis=1)._get_numeric_data()
# x_copy.fillna(x_copy.mean(), inplace=True)

# print(max([variance_inflation_factor(x_copy, i) for i in range(x_copy.shape[1])]))

# # calculate VIF
# vif = pd.DataFrame()
# vif["Variable"] = x_copy.columns
# vif["VIF"] = [variance_inflation_factor(x_copy, i) for i in range(x_copy.shape[1])]
# print(vif)

### ✏️ Your Turn — VIF Check

Uncomment and run the cell below to calculate Variance Inflation Factors. Features with VIF > 10 have severe multicollinearity and should be candidates for removal. Keep a note of which features exceed the threshold.


## Multicollinearity

* We want high correlation with target
* We don't want high correlation between features
* Drop correlated features
* Combine correlated features

Multicollinearity occurs when two or more independent variables (features) in a regression model are highly correlated, meaning they provide redundant information. While the model might still have high predictive power, multicollinearity undermines the statistical integrity of the results.


### 1. Unreliable Coefficient Estimates
The primary danger of multicollinearity is that it makes the model's coefficients ($\beta$) highly sensitive to small changes in the data.

* **The Problem:** In a linear model $Y = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + \dots + \epsilon$, the coefficient $\beta_i$ represents the change in $Y$ for a unit change in $X_i$, **holding all other variables constant**.
* **The Conflict:** If $X_1$ and $X_2$ are perfectly correlated, it is mathematically impossible to change $X_1$ while holding $X_2$ constant. This leads to unstable coefficients with high standard errors.

### 2. Reduced Statistical Power (P-values)
Because multicollinearity increases the **Variance Inflation Factor (VIF)**, the standard errors of the affected coefficients grow significantly.

* **Formula for Variance of $\hat{\beta}_j$:**
  $$\text{Var}(\hat{\beta}_j) = \frac{\sigma^2}{(n-1)\text{Var}(X_j)} \cdot \frac{1}{1-R_j^2}$$
* **The Impact:** As the correlation ($R_j^2$) between $X_j$ and other predictors approaches $1$, the variance of the estimate approaches infinity. This results in wide confidence intervals and high p-values, making it difficult to prove that a feature is statistically significant, even if it actually is.

### 3. Loss of Interpretability
For many data science projects, "Why?" is as important as "What?". Multicollinearity masks the true importance of individual features.
* If "Years of Experience" and "Age" are highly correlated, the model might assign a huge positive weight to one and a negative weight to the other, making it look like one factor is harming the outcome when it is actually helping.

### 4. Overfitting and Generalization
While multicollinearity doesn't always hurt the $R^2$ or the overall predictive accuracy on the training set, it creates a "brittle" model. An overfitted model with redundant features is less likely to generalize well to new, unseen data where the specific correlation between those two features might slightly shift.


### How to Detect and Fix Multicollinearity

| Method | Description |
| :--- | :--- |
| **VIF (Variance Inflation Factor)** | A VIF $> 5$ or $10$ indicates high multicollinearity. |
| **Feature Dropping** | Remove one of the highly correlated variables. |
| **Feature Engineering** | Combine correlated variables (e.g., $Weight / Height^2 = BMI$). |
| **Regularization** | Use **Ridge Regression** ($L_2$ regularization), which adds a penalty to the size of coefficients: $Loss = \sum (y - \hat{y})^2 + \lambda \sum \beta^2$. |
| **PCA** | Use Principal Component Analysis to transform correlated features into uncorrelated components. |

In [ ]:
# sns.jointplot(x='multicollinearity 1', y='multicollinearity 2', data=df, kind='reg', color='purple')
# plt.suptitle('Joint Plot of Multicollinear Features', y=1.02)
# plt.show()

### ✏️ Your Turn — Joint Plot (Multicollinear Pair)

Uncomment and run the cell below to visualize two multicollinear features against each other. You should see a clear linear relationship. In a comment, explain what this means for keeping both features in your model.


In [ ]:
# # iterate dropping features with high vif
# import pandas as pd
# import numpy as np
# from sklearn.preprocessing import StandardScaler
# from statsmodels.stats.outliers_influence import variance_inflation_factor

# removed=[]
# x_copy1 = x_copy.copy()
# max_vif = thresh = 10
# while max_vif >= thresh:
#   my_list = [variance_inflation_factor(x_copy1, i) for i in range(x_copy1.shape[1])]
#   max_vif = max(my_list)
#   if max_vif > thresh:
#     max_index = my_list.index(max_vif)
#     removed.append(x_copy1.columns[max_index])
#     print(x_copy1.columns[max_index], variance_inflation_factor(x_copy1, max_index))
#     x_copy1.drop(x_copy1.columns[max_index], axis=1, inplace=True)


# # Calculate VIF
# vif = pd.DataFrame()
# vif["Variable"] = x_copy1.columns
# vif["VIF"] = [variance_inflation_factor(x_copy1, i) for i in range(x_copy1.shape[1])]
# print(vif)

# # Create a variable to pickle
# data = {'vif': vif}


In [ ]:
# print(removed)

### ✏️ Your Turn — Iterative VIF Removal

Uncomment and run the two cells below to iteratively remove the highest-VIF feature until all remaining features fall below the threshold. Note the removed features — you will reference this list in Part 6.


## Outliers

Outliers are data points that differ significantly from other observations. They can be the result of variability in measurement, experimental errors, or genuinely rare events. Addressing them is a critical step in the "Data Preprocessing" phase.


### 1. Preventing Bias in Statistical Estimators
Many common statistical measures are highly sensitive to extreme values. A single outlier can shift these metrics so far that they no longer represent the "typical" data point.

* **The Mean ($\bar{x}$):** The mean is non-robust. For a dataset $X = \{x_1, x_2, \dots, x_n\}$, the mean is: $\bar{x} = \frac{1}{n} \sum_{i=1}^{n} x_i$ If one $x_i$ is extremely large, $\bar{x}$ will be pulled toward it, misrepresenting the central tendency.
* **The Variance ($\sigma^2$):** Since variance involves squaring the distance from the mean, outliers exponentially inflate the perceived spread of the data:
    $$\sigma^2 = \frac{\sum (x_i - \bar{x})^2}{n}$$



### 2. Impact on Model Training and Convergence
Machine learning models often attempt to minimize an error function. Outliers can "pull" the model's attention away from the majority of the data.

* **Linear Regression:** Ordinary Least Squares (OLS) minimizes the sum of squared residuals. An outlier with a high residual will force the regression line to tilt significantly to reduce that specific error, leading to poor fit for the rest of the data.
* **Gradient Descent:** In neural networks or logistic regression, an extreme outlier can cause massive gradients, leading to "exploding gradients" where the model weights fluctuate wildly and fail to converge.



### 3. Improving Accuracy and Generalization
If an outlier is "noise" (e.g., a sensor malfunction or a typo), keeping it in the dataset forces the model to learn a pattern that doesn't exist in reality. This leads to **Overfitting**, where the model performs perfectly on training data but fails on new, clean data.

### 4. Normalization and Scaling Issues
Many algorithms (like SVM or KNN) require features to be on the same scale.
* **Min-Max Scaling:** $$x_{scaled} = \frac{x - x_{min}}{x_{max} - x_{min}}$$
    If $x_{max}$ is an extreme outlier, all other "normal" values will be squashed into a tiny range (e.g., between $0$ and $0.01$), destroying the model's ability to distinguish between them.


### Common Strategies for Handling Outliers

| Strategy | Description | When to Use |
| :--- | :--- | :--- |
| **Trimming (Removal)** | Deleting the outlier records. | When the outlier is clearly an error (e.g., age = 200). |
| **Winsorization** | Capping the values at a specific percentile (e.g., 95th). | To limit the impact without losing data points. |
| **Transformation** | Applying $log(x)$ or Square Root. | To reduce the "distance" of outliers in skewed data. |
| **Robust Scaling** | Using Median and Interquartile Range (IQR). | When you want to keep outliers but prevent scaling issues. |

In [ ]:
# Run this cell to view a boxplot of one of your outlier columns.
# The dots beyond the whiskers are flagged outliers.
# df.boxplot(column=['outliers 1']);

### ✏️ Your Turn — Outlier Exploration

Run the boxplot cell below to visualize an outlier column. Then uncomment and run the subsequent cells to review descriptive statistics and count outliers using the IQR method across all numerical columns.

After running, add a comment noting: which columns have the most outliers, and whether you think they represent real extreme events in your story or data errors.


In [ ]:
# plt.figure(figsize=(10, 6))
# sns.violinplot(x='pd qcut1', y='target', data=df, inner='quartile')
# plt.title('Target Density by pd qcut1')
# plt.show()

In [ ]:
# Uncomment and run to see descriptive statistics for all numerical columns.
# Look at min/max for evidence of extreme values.
# df.describe()

In [ ]:
# import pandas as pd

# def count_outliers_iqr(df, column):
#     """Counts the number of outliers in a DataFrame column using the IQR method."""
#     Q1 = df[column].quantile(0.25)
#     Q3 = df[column].quantile(0.75)
#     IQR = Q3 - Q1
#     lower_bound = Q1 - 1.5 * IQR
#     upper_bound = Q3 + 1.5 * IQR
#     outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
#     return len(outliers)

# def detect_and_print_numerical_outliers_iqr(df):
#     """
#     Iterates through numerical columns in a DataFrame and prints the
#     variable name with the number of outliers based on the IQR method.
#     """
#     numerical_cols = df.select_dtypes(include=['number']).columns
#     for col in numerical_cols:
#         num_outliers = count_outliers_iqr(df, col)
#         print(f"Variable: {col}, Number of outliers (IQR): {num_outliers}")


# detect_and_print_numerical_outliers_iqr(df[df_numerical])

## Understanding Percentiles and Box Plots

In data science, we often need to look beyond the average to understand how data is distributed. Percentiles and Box Plots are the standard tools for visualizing the "spread" and "relative standing" of data points.


### 1. What are Percentiles?
A **Percentile** is a measure used in statistics indicating the value below which a given percentage of observations in a group of observations falls.

* **Definition:** The $k^{th}$ percentile is the value such that $k\%$ of the data is less than or equal to that value.
* **Median ($Q_2$):** This is the $50^{th}$ percentile. Half the data is above it, and half is below.
* **Quartiles:** We commonly divide data into four equal parts:
    * **$Q_1$ (25th Percentile):** The "lower quartile."
    * **$Q_2$ (50th Percentile):** The Median.
    * **$Q_3$ (75th Percentile):** The "upper quartile."


### 2. What is a Box Plot?
A **Box Plot** (or Box-and-Whisker Plot) is a standardized way of displaying the distribution of data based on a **five-number summary**:
1. Minimum
2. First Quartile ($Q_1$)
3. Median ($Q_2$)
4. Third Quartile ($Q_3$)
5. Maximum

#### Anatomy of a Box Plot:
* **The Box:** Spans from $Q_1$ to $Q_3$. This region contains the middle $50\%$ of the data.
* **The Median Line:** A line inside the box marking the $50^{th}$ percentile.
* **Interquartile Range (IQR):** The height (or length) of the box.
    $$IQR = Q_3 - Q_1$$
* **Whiskers:** Lines extending from the box to the smallest and largest values *within* a certain range.
* **Outliers:** Points that fall outside the whiskers. Usually defined as values:
    * Below $Q_1 - 1.5 \times IQR$
    * Above $Q_3 + 1.5 \times IQR$


### 3. Why use them together?
While percentiles give you a specific "rank," the box plot provides a visual "shape" of the data.

* **Detecting Skewness:** If the median line is closer to the bottom of the box, the data is **positively skewed** (long tail at the top). If it's closer to the top, it's **negatively skewed**.
* **Comparing Groups:** Box plots are the most efficient way to compare the distribution of a numerical variable across different categories (e.g., "Salary" vs. "Job Title").

| Feature | Statistical Value |
| :--- | :--- |
| **Lower Whisker Bound** | $Q_1 - 1.5 \times IQR$ |
| **Bottom of Box** | $25^{th}$ Percentile ($Q_1$) |
| **Middle Line** | $50^{th}$ Percentile ($Median$) |
| **Top of Box** | $75^{th}$ Percentile ($Q_3$) |
| **Upper Whisker Bound** | $Q_3 + 1.5 \times IQR$ |

## Sampling & Inference

Why the Mean Matters in Data Science

The sample mean is more than just an "average"; it is a foundational tool for statistical inference. Based on the principles of data science and statistical theory, here are the key talking points regarding its importance:

* **The Center of Gravity:** Mathematically and physically, the mean represents the balance point of a distribution. If you were to place a histogram on a pivot, the mean is the exact point where it would balance perfectly.
* **The "Smoother" Effect:** Calculating a mean can be viewed as an "equalizing" operation. It represents the value each individual in a collection would have if the total sum were redistributed evenly across all members.
* **Distribution Dependency:** The mean is determined solely by the distinct values in a collection and their relative proportions. This means any two datasets with the same underlying distribution will share the exact same mean, regardless of their size.
* **Proportions as Means:** In datasets with binary values ($0$ and $1$), the mean is mathematically equivalent to the proportion of $1$s. This is critical because it allows all mathematical properties and limit theorems of means to be applied directly to categorical proportions.
* **The Power of the Bell Shape:** A fundamental property of the sample mean is that its empirical distribution tends to become bell-shaped as the sample size increases, regardless of the shape of the population distribution.
* **A Universal Tool for Inference:** Because sample means behave predictably (forming a normal distribution) even when the population parameters are unknown, they are the primary tool used to make accurate inferences about large groups from small samples.
* **Unbiased Estimation:** The sample mean serves as an unbiased estimate of the population mean. As your sample size $n$ increases, the variability of this estimate decreases, leading to higher precision in your model.


## 10 Essential EDA Visualizations

The ten visualizations below are provided as reference code. **Your job is to run each one using your story's variable names.**

For each visualization:
1. Copy the code into a new code cell directly below its description.
2. Replace the placeholder column names (e.g., `'target'`, `'informative_1'`) with your renamed story variables.
3. Run the cell.
4. Add a one-sentence comment in the code cell describing what the chart reveals about your historical event.

**Prerequisites — run this first:**
```python
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df['date'] = pd.to_datetime(df['date'])
```


### 1. Histogram & KDE (Univariate Distribution)
Used to visualize the distribution of a single numerical variable, such as `target`. It helps identify skewness and kurtosis ($\gamma_2$).
```python
plt.figure(figsize=(10, 6))
sns.histplot(df['target'], kde=True, color='blue')
plt.title('Distribution of Target Variable')
plt.xlabel('Target Value')
plt.ylabel('Frequency')
plt.show()
```

### 2. Box Plot (Outliers and Quartiles)
Visualizes the five-number summary and identifies outliers based on the Interquartile Range ($IQR = Q_3 - Q_1$). Here, we look at `informative_1` across different `class` categories.
```python
plt.figure(figsize=(10, 6))
sns.boxplot(x='class', y='informative_1', data=df, palette='Set2')
plt.title('Informative_1 Distribution by Class')
plt.show()
```

### 3. Count Plot (Categorical Frequency)
The standard way to check for class imbalance in categorical variables like `categorical_1`.
```python
plt.figure(figsize=(10, 6))
sns.countplot(x='categorical_1', data=df, order=df['categorical_1'].value_counts().index)
plt.title('Frequency of Categorical_1 Levels')
plt.xticks(rotation=45)
plt.show()
```

### 4. Scatter Plot (Numerical Relationships)
Used to detect correlations or clusters between two continuous variables, such as `informative_1` and `target`.
```python
plt.figure(figsize=(10, 6))
sns.scatterplot(x='informative_1', y='target', hue='class', data=df, alpha=0.6)
plt.title('Informative_1 vs Target (Colored by Class)')
plt.show()
```

### 5. Correlation Heatmap (Multicollinearity)
A graphical representation of the correlation matrix where each cell represents the Pearson coefficient ($\rho$). Crucial for identifying redundant features.
```python
plt.figure(figsize=(12, 10))
numeric_df = df.select_dtypes(include=['float64', 'int64'])
sns.heatmap(numeric_df.corr(), annot=True, fmt=".2f", cmap='coolwarm', center=0)
plt.title('Correlation Matrix Heatmap')
plt.show()
```

### 6. Pair Plot (Multivariate Overview)
Generates a grid of scatter plots and histograms for a subset of variables. Excellent for a quick overview of relationships.
```python
# Selecting a subset of relevant columns to avoid performance lag
cols = ['informative_1', 'informative_2', 'target', 'class']
sns.pairplot(df[cols], hue='class', diag_kind='kde')
plt.show()
```

### 7. Violin Plot (Density Estimation by Category)
Combines a box plot with a Kernel Density Estimate (KDE). It shows the peakiness and distribution of `target` across different `pd qcut1` categories.
```python
plt.figure(figsize=(10, 6))
sns.violinplot(x='pd qcut1', y='target', data=df, inner='quartile')
plt.title('Target Density by pd qcut1')
plt.show()
```

### 8. Bar Plot (Aggregated Statistics)
Unlike a count plot, this displays an aggregate statistic (like the mean $\mu$) of a numerical variable for each category.
```python
plt.figure(figsize=(10, 6))
sns.barplot(x='class', y='correlated w target 1', data=df, estimator='mean', errorbar='sd')
plt.title('Mean of Correlated_w_target_1 per Class (with Standard Deviation)')
plt.show()
```

### 9. Line Plot (Time Series Trends)
Since your data contains a `date` column, a line plot is essential to observe how the `target` variable changes over time $t$.
```python
plt.figure(figsize=(12, 6))
df_time = df.sort_values('date')
sns.lineplot(x='date', y='target', data=df_time)
plt.title('Target Variable Trend Over Time')
plt.xticks(rotation=45)
plt.show()
```

### 10. Joint Plot (Bivariate + Marginal Distributions)
Provides a scatter plot of two variables along with their individual histograms on the axes. Useful for seeing how $X$ and $Y$ relate while simultaneously seeing their individual spreads.
```python
sns.jointplot(x='multicollinearity 1', y='multicollinearity 2', data=df, kind='reg', color='purple')
plt.suptitle('Joint Plot of Multicollinear Features', y=1.02)
plt.show()
```


In [ ]:
# Run this cell to save your EDA-stage dataframe before moving to Part 4.
# Do not skip this step — Part 4 loads from this file.
df.to_csv('data science fiction iv pt 2.csv', index=False)

# Part 4 - Data Prep

In this part you will clean the dataset by removing or correcting problems that were deliberately introduced in Part 1. Work through each section below **in order** — some steps depend on the previous ones being complete.

Reference: https://www.udemy.com/course/feature-engineering-for-machine-learning

The cleaning steps you need to complete:

1. **Constants** — drop columns where every value is identical (zero variance)
2. **Quasi-constants** — drop columns where one value dominates (>98% the same)
3. **Duplicate rows** — identify and remove exact duplicate rows
4. **Duplicate features** — identify and remove columns that are identical to another column
5. **Missing data** — decide on an imputation strategy for each column with nulls
6. **Scaling** — apply the appropriate scaler to columns that need it
7. **Outliers** — decide on a treatment strategy for columns with extreme values

> **Reminder:** Use your story variable names throughout, not the original system names. Your final cleaned dataset should be ready for encoding and modeling.


## Load Data

In [ ]:
# import pandas as pd

# df = pd.read_csv('data science fiction iv pt 2.csv')
# print(df.shape)
# print(df.info())
# df.head()

## Clean the Data

Complete each cell below. Each placeholder comment names the cleaning task. Write your solution code directly in the cell. Run `df.shape` or `df.info()` after major steps to confirm the change took effect.


In [ ]:
# CONSTANTS
# Columns where every row has the same value carry zero information and must be removed.
#
# Strategy:
#   1. Use df.nunique() to find columns with only 1 unique value.
#   2. Store the list of constant column names.
#   3. Drop them from df using df.drop().
#
# Hint: df.nunique()[df.nunique() == 1].index gives you the constant columns directly.
#
# How many constant columns did you find? Note it here: ___


In [ ]:
# QUASI-CONSTANTS
# Columns where one value makes up nearly all rows (e.g., >98%) add almost no signal.
#
# Strategy:
#   1. For each categorical/object column, compute value_counts(normalize=True).
#   2. If the top category's proportion exceeds your threshold (suggest 0.98), flag the column.
#   3. Drop the flagged columns from df.
#
# Hint: series.value_counts(normalize=True).iloc[0] gives the top proportion.
#
# How many quasi-constant columns did you find? Note it here: ___


In [ ]:
# DUPLICATE ROWS
# Identical rows add no new information and can bias your model.
#
# Strategy:
#   1. Use df.duplicated().sum() to count duplicates.
#   2. Use df.drop_duplicates(inplace=True) to remove them.
#   3. Reset the index: df.reset_index(drop=True, inplace=True)
#
# How many duplicate rows were removed? Note it here: ___


In [ ]:
# DUPLICATE FEATURES
# Columns that are identical to another column are redundant.
#
# Strategy:
#   1. Transpose the dataframe and use .duplicated() to find duplicate columns.
#   2. Drop one from each duplicate pair.
#
# Hint: df.T.duplicated() returns a boolean mask over column names.
#       df.columns[df.T.duplicated()].tolist() gives you the duplicates to drop.
#
# Which columns were duplicates? Note them here: ___


In [ ]:
# MISSING DATA
# Handle null values column by column. Your strategy should depend on the variable type
# and how much data is missing.
#
# General rules:
#   - Drop columns with > 60% missing (too little data to impute reliably).
#   - For numerical columns: impute with mean or median.
#   - For categorical columns: impute with mode or a placeholder label like 'Unknown'.
#
# Steps:
#   1. Run df.isnull().sum() / len(df) * 100 to see % missing per column.
#   2. Drop columns that exceed your threshold.
#   3. Impute the rest using df.fillna() or SimpleImputer.
#
# Verify: after imputation, df.isnull().sum().sum() should return 0.


In [ ]:
# SCALING
# Two columns were generated with distributions that need scaling before modeling.
# Refer to your story mapping to identify which columns these are
# (look for the ones generated by gen_standard_scaling and gen_minmax_scaling in Part 1).
#
# Standard Scaling (StandardScaler): centers to mean=0 and std=1.
#   - Use when the feature is approximately normal.
#
# Min-Max Scaling (MinMaxScaler): compresses values into [0, 1].
#   - Use when the range of values matters and outliers are already handled.
#
# Steps:
#   1. Import the appropriate scaler(s) from sklearn.preprocessing.
#   2. Fit and transform the target columns.
#   3. Replace the original columns in df.
#
# Note: Technically, scalers should be fit on training data only (after the
# train/test split in Part 6). For now, apply to the full df.


In [ ]:
# OUTLIERS
# You identified outlier-heavy columns in Part 3. Decide how to handle each one.
#
# Choose a strategy based on the column and your story:
#   - Winsorization: cap values at the 5th and 95th percentile.
#       Use when you want to limit the impact of extremes without losing rows.
#       Example: df[col] = df[col].clip(lower=df[col].quantile(0.05),
#                                        upper=df[col].quantile(0.95))
#
#   - Trimming: remove rows where the value is beyond ±3 std devs.
#       Use only if the outliers are clearly data errors (not real events).
#
#   - Log transformation: apply np.log1p() to compress extreme values.
#       Use for right-skewed data where outliers are real but extreme.
#
# For each outlier column, add a comment explaining:
#   - Which strategy you chose
#   - Whether the outlier represents a real event in your story or a data error


## Identify Variable Types for Encoding

In [ ]:
# # Run this cell to re-identify variable types after cleaning.
# # The column set has changed — this gives you an updated view before encoding.

# df_numerical = df.select_dtypes(include='number').columns
# df_object = df.select_dtypes(include=['object']).columns
# df_discrete = df.select_dtypes(include=['category']).columns
# df_categorical_features = df.select_dtypes(include=['category', 'object']).columns
# print('Numerical:', list(df_numerical))
# print('Object:', list(df_object))
# print('Discrete/Category:', list(df_discrete))

# Part 5 - Feature Engineering

In this part you will:
1. **Create derived variables** — new columns computed from existing ones.
2. **Encode categorical variables** — convert text/category columns into numbers the model can use.

After this part, every column in `df` (except `class`) must be numerical. The final cell saves the result for Part 6.


## Derived Variables

A derived variable is a new feature you compute by combining or transforming existing columns. The goal is to capture a relationship that the raw columns might not express on their own.

**Your task:** Create at least **two** derived variables that make sense in the context of your historical story.

**Common strategies:**

| Technique | Example Code | When to Use |
| :--- | :--- | :--- |
| Ratio | `df['risk_ratio'] = df['col_a'] / (df['col_b'] + 1e-9)` | When the relationship between two quantities matters more than their individual values |
| Interaction term | `df['interaction'] = df['col_a'] * df['col_b']` | When two features jointly affect the outcome |
| Binning | `pd.cut(df['col'], bins=3, labels=['Low','Med','High'])` | To convert a continuous feature into an ordered category |
| Date component | `df['year'] = pd.to_datetime(df['date']).dt.year` | To extract meaningful parts of a date |

**For each derived variable you create, add a comment explaining:**
- Which columns it combines
- What it represents in your historical story
- Why you think it might be predictive of the outcome (`class`)


In [ ]:
# DERIVED VARIABLES
# Create at least two new columns derived from existing ones.
# Use your story mapping column names.
#
# Example structure:
#   df['your_derived_name_1'] = df['col_a'] / (df['col_b'] + 1e-9)  # ratio — what it means
#   df['your_derived_name_2'] = df['col_a'] * df['col_b']            # interaction — what it means
#
# Add a comment for each explaining what it represents in your story.


## Categorical Encoding

Machine learning models require numerical input. Any remaining object or category columns must be encoded before modeling.

| Situation | Recommended Encoding |
| :--- | :--- |
| 2 unique values (binary) | `df['col'].map({'Label_A': 0, 'Label_B': 1})` |
| 3–10 unique values (no natural order) | `pd.get_dummies(df, columns=['col'], drop_first=True)` |
| 3+ values with natural order (Low/Med/High) | `OrdinalEncoder` with defined category order |
| 10+ unique values or not useful for modeling | Drop the column |

**Your task:**
1. Look at the output of the variable-type cell above to see which columns still need encoding.
2. Apply the appropriate encoding to each one.
3. Drop any columns that cannot be meaningfully encoded (e.g., raw name fields, free-text fields).


In [ ]:
# CATEGORICAL ENCODING
# Encode all remaining object/category columns.
# Choose the method based on the number of unique values (cardinality).
#
# Common patterns:
#   Binary:   df['col'] = df['col'].map({'Label_A': 0, 'Label_B': 1})
#
#   One-hot:  df = pd.get_dummies(df, columns=['col'], drop_first=True)
#
#   Ordinal:  from sklearn.preprocessing import OrdinalEncoder
#             enc = OrdinalEncoder(categories=[['Low', 'Med', 'High']])
#             df[['col']] = enc.fit_transform(df[['col']])
#
#   Drop:     df.drop(columns=['name_col', 'free_text_col'], inplace=True)
#
# Leave a comment for each column noting which method you used and why.


In [ ]:
# # Verify that all columns are now numerical (except 'class').
# # If any object columns remain, go back and encode or drop them before continuing.
# non_numeric = df.drop('class', axis=1).select_dtypes(exclude='number').columns.tolist()
# if non_numeric:
#     print('⚠️  These columns still need encoding or removal:', non_numeric)
# else:
#     print('✅ All columns are numerical. Ready for feature selection.')
# df.info()

In [ ]:
# # Save your fully encoded dataframe. Part 6 loads from this file.
# df.to_csv('data science fiction iv pt 3.csv', index=False)

# Part 6 - Feature Selection

Feature selection identifies which variables actually help your model generalize. With a dataset this large, you are looking to reduce noise and multicollinearity before modeling.

You will apply **five selection methods** below and compare their recommendations. At the end of this section you will combine them into a final feature list.

Feature selection methods fall into three categories:

## 1. Filter Methods
Select features based on statistical properties, independently of any model. Fast and computationally efficient.
- Examples: Pearson Correlation (`corrwith`), Mutual Information, SelectKBest.

## 2. Wrapper Methods
Evaluate feature subsets by training a model on each one and measuring performance. More accurate but expensive.
- Examples: Recursive Feature Elimination (RFE).

## 3. Embedded Methods
Feature selection happens during model training — the algorithm rewards or penalizes features based on their contribution.
- Examples: SelectFromModel (L1/L2 regularization), Random Forest Importance.


In [ ]:
# import pandas as pd

# df = pd.read_csv('data science fiction iv pt 3.csv')
# print(df.shape)
# print(df.info())
# df.head()

## Train Test Split

random_state was initialized in the first code cell

In [ ]:
# from sklearn.model_selection import train_test_split

# # random_state was set in Part 1 — it ensures your split is reproducible.
# X_train, X_test, y_train, y_test = train_test_split(
#     df.drop('class', axis=1), df['class'],
#     test_size=0.3, random_state=random_state
# )
# print('X_train:', X_train.shape, '  X_test:', X_test.shape)

## CorrWith — Correlation with the Target

`DataFrame.corrwith()` computes the pairwise correlation between each column in the dataframe and a target Series. This is a quick filter-method starting point: a higher |ρ| with `class` suggests a feature is more likely to be a useful predictor.

Run the cell below, then note the top 5 features by absolute correlation in the comment. You will compare this list with the other four methods at the end of Part 6.


In [ ]:
# # CorrWith — correlation of each feature with the target variable
# corr_with_target = X_train.corrwith(y_train).abs().sort_values(ascending=False)
# print(corr_with_target)

# # Note your top 5 features here:
# # 1.
# # 2.
# # 3.
# # 4.
# # 5.

## Mutual Information

### ✏️ Your Turn — Mutual Information

Uncomment and run the two cells below. Mutual Information measures how much knowing a feature reduces uncertainty about the class label. It captures non-linear relationships that Pearson correlation misses.

After running, note the top 5 features in a comment and compare them to your CorrWith results.


In [ ]:
# # mutual information
# import matplotlib.pyplot as plt
# from sklearn.feature_selection import mutual_info_classif

# mi = mutual_info_classif(X_train, y_train)
# mi = pd.Series(mi)
# mi.index = X_train.columns
# mi.sort_values(ascending=False).plot.bar()
# plt.ylabel('Mutual Information');

In [ ]:
# mi_keepers = mi.sort_values(ascending=False).index[:5]
# print(mi_keepers)

## SelectKBest

### ✏️ Your Turn — SelectKBest

Uncomment and run the cell below. SelectKBest applies a statistical test (`f_classif`) to score each feature and keeps the top k. Compare results to the two filter methods above.


In [ ]:
# # SelectKBest
# from sklearn.feature_selection import SelectKBest, f_regression, f_classif

# selector = SelectKBest(f_classif, k=5) # Select the top 5 features
# X_new = selector.fit(X_train, y_train)

# kb_keepers = X_train.columns.values[selector.get_support()]
# print(kb_keepers)

## Select From Model

### ✏️ Your Turn — Select From Model (Embedded)

Uncomment and run the cell below. This method trains a logistic regression with regularization and keeps only features with non-negligible coefficients. Because selection happens during training, this is an embedded method.


In [ ]:
# # Select from model
# import numpy as np
# from sklearn.linear_model import LinearRegression, LogisticRegression
# from sklearn.feature_selection import SelectFromModel
# from sklearn.preprocessing import StandardScaler

# scaler = StandardScaler()
# scaler.fit(X_train)
# X_scaled = scaler.transform(X_train)

# selections = SelectFromModel(estimator=LogisticRegression()).fit(X_scaled, y_train)
# mt_keepers = X_train.columns.values[selections.get_support()]
# print(mt_keepers)

## Recursive Feature Elimination (RFE)

In [ ]:
# from sklearn.feature_selection import RFE
# from sklearn.linear_model import LinearRegression, LogisticRegression

# estimator = LogisticRegression()
# selector = RFE(estimator, n_features_to_select=5) # Select the top 5 features
# X_new = selector.fit_transform(X_scaled, y_train)
# rf_keepers = X_train.columns.values[selections.get_support()]
# print(rf_keepers)

### ✏️ Your Turn — RFE (Wrapper Method)

Uncomment and run the cell below. RFE trains the model repeatedly, removing the weakest feature each iteration. It is the most thorough of the five methods. This may take a few seconds to run.


## Random Forest Importance


In [ ]:
# # random forest importance
# from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
# from sklearn.feature_selection import SelectFromModel

# selects = SelectFromModel(RandomForestClassifier(n_estimators=100, random_state=random_state), max_features=4)
# selects.fit(X_train, y_train)
# rfi = X_train.columns[(selects.get_support())]
# rfi.tolist()

### ✏️ Your Turn — Random Forest Importance (Embedded)

Uncomment and run the cell below. Random Forest ranks features by how much they reduce impurity across all trees. It handles non-linear relationships well and is less sensitive to scaling than logistic regression.


## Build Your Final Feature List

You have now run five selection methods. Before finalizing your features, also revisit findings from Parts 3 and 4:

- **Correlated feature pairs** — which had |ρ| > 0.7? Keep at most one from each pair.
- **VIF** — which variables were flagged for multicollinearity?
- **Outlier columns** — were any columns too noisy to trust?

**Your task — complete this before filling in the code cells below:**

Create a summary table comparing which features appeared across selection methods:

| Feature | CorrWith | Mutual Info | SelectKBest | SelectFromModel | RFE | Random Forest | Total |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: | :---: |
| *feature name* | ✓ | ✓ | | ✓ | | ✓ | 3 |

Use this table to build your `features_to_model` list. A feature appearing in **3 or more methods** is a strong candidate. Also consider your story context — does the feature make narrative sense as a predictor of the catastrophe?

Aim for **5–8 features**. Too few → underfitting. Too many → overfitting.


In [ ]:
# # Build your final feature list based on the comparison table above.
# # Features that appeared in multiple selection methods AND make story sense are the best choices.
# # Aim for 5-8 features.

# features_to_model = [
#     # 'your_feature_1',
#     # 'your_feature_2',
#     # 'your_feature_3',
#     # ...add more as needed
# ]

# print(f'{len(features_to_model)} features selected:', features_to_model)

In [ ]:
# # Filter your train and test sets to only the selected features.
# X_train = X_train[features_to_model]
# X_test = X_test[features_to_model]
# print('Final X_train shape:', X_train.shape)
# print('Final X_test shape:', X_test.shape)

# Part 7 - Data Modeling and Evaluation

You have cleaned, engineered, and selected your features. Now you will train a logistic regression model and evaluate it thoroughly.

**This part has both code tasks and written explanation tasks.** The explanation cells are graded — write in complete sentences and connect your answers back to your historical story.


## Logistic Regression

In [ ]:
# # Train your logistic regression model, generate predictions, and print accuracy.
# from sklearn.linear_model import LogisticRegression
# from sklearn.metrics import confusion_matrix, accuracy_score

# model = LogisticRegression(solver='liblinear', random_state=random_state)
# model.fit(X_train, y_train)
# predictions = model.predict(X_test)

# train_accuracy = model.score(X_train, y_train)
# test_accuracy = model.score(X_test, y_test)
# print(f'Training Accuracy: {train_accuracy:.4f}')
# print(f'Testing Accuracy:  {test_accuracy:.4f}')

## Model Evaluation

### Confusion Matrix and Accuracy

In [ ]:
# from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# tn, fp, fn, tp = confusion_matrix(y_test, predictions).ravel()
# print('accuracy:', accuracy_score(y_test, predictions))

**The order of `y_test` (true labels) and `predictions` (predicted labels) matters significantly for `confusion_matrix` and `classification_report` in scikit-learn.**

However, for **`accuracy_score`**, the order does **not** matter because it simply calculates the proportion of correctly classified instances, regardless of which is considered the "true" and which is the "predicted" set in the function call.

Let's break down why the order is crucial for `confusion_matrix` and `classification_report`:

**`confusion_matrix(y_test, predictions)`:**

* The first argument (`y_test`) should always be the **true labels** (the actual values).
* The second argument (`predictions`) should always be the **predicted labels** (the values your model has outputted).

The output of `confusion_matrix` is a 2x2 (for binary classification) or NxN (for multi-class classification) array where:

* The rows correspond to the **true classes**.
* The columns correspond to the **predicted classes**.

Therefore, `confusion_matrix(y_test, predictions)` will produce a matrix where:

* `TN` (True Negative) is the count of instances where the true label was negative and the prediction was negative.
* `FP` (False Positive) is the count of instances where the true label was negative and the prediction was positive.
* `FN` (False Negative) is the count of instances where the true label was positive and the prediction was negative.
* `TP` (True Positive) is the count of instances where the true label was positive and the prediction was positive.

**`accuracy_score(y_test, predictions)`:**

* For `accuracy_score`, the order does **not** matter. Accuracy is calculated as the number of correct predictions divided by the total number of predictions:

    `Accuracy = (Number of Correct Predictions) / (Total Number of Predictions)`

    Whether you compare `y_test` against `predictions` or `predictions` against `y_test`, the set of correctly matched instances will be the same, and the total number of instances remains the same. Therefore, the accuracy score will be identical regardless of the order of the arguments.



In [ ]:
# from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# acc = accuracy_score(y_test, predictions)
# print(f'Overall Accuracy: {acc:.4f}')

# cm = confusion_matrix(y_test, predictions)
# if cm.size == 4:
#     tn, fp, fn, tp = cm.ravel()
#     print(f'Confusion Matrix: TN={tn}, FP={fp}, FN={fn}, TP={tp}')
# else:
#     print('Confusion Matrix (Multi-class):\n', cm)

# print('\nClassification Report:')
# print(classification_report(y_test, predictions))

# report_dict = classification_report(y_test, predictions, output_dict=True)

**`classification_report(y_test, predictions)`:**

* Similar to `confusion_matrix`, the first argument (`y_test`) must be the **true labels**, and the second argument (`predictions`) must be the **predicted labels**.

The `classification_report` provides a text summary of the precision, recall, F1-score, and support for each class. These metrics are calculated based on the true positives, true negatives, false positives, and false negatives, which are directly derived from the correct alignment of true and predicted labels. Swapping the order would lead to incorrect calculations of these metrics for each class.

### The Classification Report

#### 1. The Core Metrics (Per-Class)
These metrics are calculated for each individual class (e.g., class `0` and class `1`).

* **Precision (The "Quality" Metric):**
    This answers: *Of all the instances the model predicted as positive, how many were actually positive?*
    $$\text{Precision} = \frac{TP}{TP + FP}$$
    * **High Precision** means the model doesn't cry wolf often (few False Positives).
* **Recall (The "Quantity" Metric):**
    This answers: *Of all the actual positive instances that exist, how many did the model find?*
    $$\text{Recall} = \frac{TP}{TP + FN}$$
    * **High Recall** means the model is great at "finding the needle in the haystack" (few False Negatives).
* **F1-Score (The "Balance" Metric):**
    The harmonic mean of Precision and Recall. It’s useful because it penalizes extreme values. If you have a Precision of 1.0 but a Recall of 0.0, your F1-score will be 0.
    $$F_1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$
* **Support:**
    This is simply the **count** of actual occurrences of the class in your specified dataset ($y\_test$). If the support is very uneven (e.g., 900 for class A and 10 for class B), your dataset is imbalanced.

#### 2. The Averages (The "Global" View)
The bottom of the report provides three different ways to look at the model's overall health:

| Metric | How it's calculated | When to use it |
| :--- | :--- | :--- |
| **Accuracy** | Total correct / Total samples | When your classes are **balanced**. |
| **Macro Avg** | Arithmetic mean of metrics (e.g., $(P_{class0} + P_{class1}) / 2$) | When you want to treat all classes as **equally important**, regardless of size. |
| **Weighted Avg** | Mean weighted by the **Support** of each class. | When you want to account for **class imbalance** (gives more "weight" to the majority class). |

#### Why does this matter?
A model can have **99% Accuracy** but a **0% Recall** for the minority class (e.g., detecting a rare disease). The classification report is designed to expose that "fake" success by showing you exactly where the model is failing—is it misidentifying things (low precision) or missing them entirely (low recall)?

**In summary:**

* **`confusion_matrix`:** **Order matters.** Always use `confusion_matrix(y_test, predictions)`.
* **`classification_report`:** **Order matters.** Always use `classification_report(y_test, predictions)`.
* **`accuracy_score`:** **Order does not matter.** `accuracy_score(y_test, predictions)` will yield the same result as `accuracy_score(predictions, y_test)`.

It's crucial to maintain the correct order of true labels and predicted labels when using `confusion_matrix` and `classification_report` to ensure accurate evaluation of your classification model.

## Metrics

* tn = pred 0 actual 0
* fp = pred 1 actual 0
* fn = pred 0 actual 1
* tp = pred 1 actual 1
* acc(uracy) = $\frac{tn + tp}{total}$
* error = $\frac{fp + fn}{total}$
* prev(alence) = $\frac{fn + tp}{total}$
* queue = $\frac{fp + tp}{total}$
* tpr = $\frac{tp}{tp + fn}$
    * true positive rate
    * recall
    * sensitivity
    * prob of detection
    * 1 - fnr
* fnr = $\frac{fn}{tp + fn}$
    * false negative rate
    * type II error
    * 1 - tpr
* tnr = $\frac{tn}{tn + fp}$
    * true negative rate
    * specificity
    * 1 - fpr
* fpr = $\frac{fp}{tn + fp}$
    * false positive rate
    * type I error
    * fall out
    * prob of false claim
    * 1 - tnr
* ppv = $\frac{tp}{tp + fp}$
    * positive predicted value
    * precision
    * 1 - fdr
* fdr = $\frac{fp}{tp + fp}$
    * false discovery rate
    * 1 - ppv
* npv = $\frac{tn}{tn + fn}$
    * negative predicted value
    * 1 - for
* for = $\frac{fn}{tn + fn}$
    * false omission rate
    * 1 - npv
* liklihood ratio+ (lr+) = $\frac{tpr}{fpr}$
    * roc
* liklihood ratio- (lr-) = $\frac{fnr}{tnr}$
* diagnostic odds ratio = $\frac{lr+}{lr-}$
* f1 score = 2 * $\frac{precision-recall}{precision+recall}$
* Youden's J = sensitivity + specificity - 1 = tpr - fpr
* Matthew's Correlation Coefficient = $\frac{(tp*tn)-(fp*tp)}{\sqrt{(tp+fp)(tp+fn)(tn+fp)(tn+fn)}}$
  

## Confusion Matrix

In [ ]:
# print(confusion_matrix(y_test, predictions))

### Explanation — Confusion Matrix

**Answer the following questions in complete sentences. Connect your answers to your historical story.**

1. How many True Positives and True Negatives did your model produce? What do these represent in your historical scenario?

2. Which error type did your model make more of — False Positives or False Negatives? Translate this into story terms (e.g., "The model raised the alarm when there was no plague X times, and missed an actual plague outbreak Y times.").

3. Referring back to Part 2's hypothesis framework: did your model make more Type I errors (false alarms) or Type II errors (missed catastrophes)? Given the stakes of your historical scenario, which error type is more costly, and why?

---
*Write your response here.*


## Precision Recall

In [ ]:
# print(classification_report(y_test, predictions))

### Explanation — Precision and Recall

**Answer the following questions in complete sentences. Connect your answers to your historical story.**

1. What is the **precision** for class 1 (the catastrophic event)? In plain terms: out of every time your model predicted a catastrophe, what fraction was it correct?

2. What is the **recall** for class 1? In plain terms: out of every actual catastrophe in the test set, what fraction did your model catch?

3. Is your F1-score closer to your precision or your recall? What does that imply about which type of error your model makes more often?

4. Given your story's stakes, should this model optimize for **precision** or **recall**? (e.g., "In a plague scenario, missing a real outbreak is more dangerous than a false alarm, so I would prioritize recall.") Justify your answer in 2–3 sentences.

---
*Write your response here.*


## Bias-Variance Decomposition

Bias-variance decomposition breaks your model's total error into three parts:
- **Bias** — error from overly simplistic assumptions (underfitting: model misses the signal)
- **Variance** — error from over-sensitivity to the training data (overfitting: model memorized noise)
- **Noise** — irreducible error inherent to the data

A well-calibrated model balances bias and variance. If bias dominates, the model is too simple. If variance dominates, the model has overfit.

**Before running the cell below, install the required library:**
```python
pip install mlxtend -q
```


In [ ]:
# # pip install mlxtend -q  # Uncomment and run this line first if mlxtend is not installed.

# from mlxtend.evaluate import bias_variance_decomp

# avg_expected_loss, avg_bias, avg_var = bias_variance_decomp(
#     model,
#     X_train.values,
#     y_train.values,
#     X_test.values,
#     y_test.values,
#     loss='0-1_loss',
#     random_seed=random_state)

# print('Average expected loss: %.3f' % avg_expected_loss)
# print('Average bias:          %.3f' % avg_bias)
# print('Average variance:      %.3f' % avg_var)

### Explanation — Bias-Variance

**Answer the following questions in complete sentences.**

1. Report your model's bias, variance, and expected loss values.

2. Is **bias** or **variance** the larger contributor to your model's error? What does that tell you — is the model underfitting or overfitting?

3. Compare your training accuracy and test accuracy from the Logistic Regression cell. Does the gap between them align with what the bias-variance result is telling you? Explain why or why not.

4. If you wanted to reduce the dominant source of error, what is **one concrete change** you could make to your model or feature set? (e.g., add more features, reduce features, change the regularization strength, collect more data.)

---
*Write your response here.*
